# Проект: семантический поиск товаров по текстовому запросу

**Ваша роль.** Вы специалист по Data Science. Компания развивает онлайн-каталог товаров, и сейчас руководству кажется, что классический поиск по ключевым словам плохо справляется с реальными запросами. Пользователи пишут названия с ошибками, кириллицей, используют просторечные слова, ищут товары по назначению, а не по функциональности («удобные кроссовки для бега» вместо «мужские спортивные кроссовки с амортизацией»). Команда хочет проверить, поможет ли векторный поиск улучшить качество поисковой системы.

**Задача.** Вам нужно построить MVP системы семантического поиска, сравнить его с лексической baseline-моделью на подготовленной разметке и дать бизнесу обоснованную рекомендацию о том, где новое решение помогает, где ошибается и стоит ли его вообще внедрять.

## Данные

| Файл | Что это | Когда нужен |
|---|---|---|
| wb_products_raw_sample.parquet | Исходный каталог, ~200 тысяч строк, все категории. <br> Вам нужно самостоятельно отобрать рабочее подмножество этих данных | Недели 1–2 |
| wb_products_dedup.parquet | Пул асессоров: множество товаров, по которому делалась разметка | Неделя 2, раздел 4 |
| queries_synthetic_train.parquet | Разметка релевантности, train | Недели 2–3 |
| queries_synthetic_test.parquet | Разметка релевантности, test | **один прогон** в конце недели 3 |

Файлы с разметкой содержат четыре столбца: `query_id`, `query_text`, `item_id`, `relevance`. Шкала в `relevance` от `0` до `3`, где `0` — товар нерелевантен, а `3` — максимально релевантен. Связь с каталогом: `item_id` ↔ `imt_id`.

In [ ]:
from pathlib import Path
import html
import random
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

DATA_DIR = Path("data") if Path("data").exists() else Path(".")
RAW_PRODUCTS_PATH = DATA_DIR / "wb_products_raw_sample.parquet"
ASSESSED_POOL_PATH = DATA_DIR / "wb_products_dedup.parquet"
TRAIN_QUERIES_PATH = DATA_DIR / "queries_synthetic_train.parquet"
TEST_QUERIES_PATH = DATA_DIR / "queries_synthetic_test.parquet"


---

## Неделя 1 — знакомство с данными и EDA

**Цель** — понять, с какими данными работаете, чтобы принять обоснованные решения:

- выбрать категории, с которыми будете работать;
- определить, как будете очищать описания от всего лишнего;
- решить, из каких полей собирать текст товара.

**Главный критерий успеха:** любое ваше решение по данным вы можете подкрепить ссылкой на конкретные результаты анализа.

### 1. Загрузка данных

Загрузите каталог и обе разметки. Изучите их структуру: проанализируйте колонки и типы данных в них, определите, что описывает одна строка.

**Подсказка.** `imt_id` — карточка товара, `nm_id` — конкретный артикул этого товара, то есть его разновидность, отличающаяся цветом, размером или чем-то ещё. Определите, что из этого должен возвращать поиск, — от этого зависит, на каком уровне вы будете строить индекс.

In [ ]:
df_raw = pd.read_parquet(RAW_PRODUCTS_PATH)
assessed_pool = pd.read_parquet(ASSESSED_POOL_PATH)
queries_train = pd.read_parquet(TRAIN_QUERIES_PATH)
queries_test = pd.read_parquet(TEST_QUERIES_PATH)

overview = pd.DataFrame({
    "rows": [len(df_raw), len(assessed_pool), len(queries_train), len(queries_test)],
    "columns": [df_raw.shape[1], assessed_pool.shape[1], queries_train.shape[1], queries_test.shape[1]],
    "duplicates": [df_raw.duplicated().sum(), assessed_pool.duplicated().sum(),
                   queries_train.duplicated().sum(), queries_test.duplicated().sum()],
}, index=["raw_products", "assessed_pool", "queries_train", "queries_test"])
display(overview)
display(df_raw.head(3))
display(queries_train.head(3))


**Проверьте связность данных:** все ли `item_id` из разметки есть в каталоге, нет ли пересечения `query_id` между `train` и `test`.

In [ ]:
train_ids = set(queries_train["query_id"])
test_ids = set(queries_test["query_id"])
train_texts = set(queries_train["query_text"].str.strip().str.lower())
test_texts = set(queries_test["query_text"].str.strip().str.lower())

print("Пересечение query_id", len(train_ids & test_ids))
print("Пересечение текстов", len(train_texts & test_texts))
print("Товары train вне сырого каталога", (~queries_train["item_id"].isin(df_raw["imt_id"])).sum())
print("Товары test вне сырого каталога", (~queries_test["item_id"].isin(df_raw["imt_id"])).sum())


### 2. EDA

In [ ]:
text_columns = ["description", "nm_colors_names", "brand_name", "subj_name"]
empty_share = {}
for column in text_columns:
    values = df_raw[column].fillna("").astype(str).str.strip()
    empty_share[column] = values.eq("").mean()

print("Строк в каталоге", len(df_raw))
print("Уникальных imt_id", df_raw["imt_id"].nunique())
print("Уникальных nm_id", df_raw["nm_id"].nunique())
print("Повторных imt_id", df_raw.duplicated("imt_id").sum())
display(pd.Series(empty_share, name="empty_share").to_frame())

lengths = pd.DataFrame({
    "name_words": df_raw["imt_name"].fillna("").astype(str).str.split().str.len(),
    "description_words": df_raw["description"].fillna("").astype(str).str.split().str.len(),
    "description_chars": df_raw["description"].fillna("").astype(str).str.len(),
})
display(lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df_raw["subj_root_name"].value_counts().head(15).sort_values().plot.barh(ax=axes[0])
axes[0].set_title("Крупнейшие корневые категории")
axes[0].set_xlabel("Число строк")
sns.histplot(lengths["description_words"].clip(upper=lengths["description_words"].quantile(0.99)),
             bins=40, ax=axes[1])
axes[1].set_title("Длина описаний без верхнего 1 процента")
axes[1].set_xlabel("Число слов")
plt.tight_layout()
plt.show()

display(queries_train["relevance"].value_counts(normalize=True).sort_index().rename("share").to_frame())
print("Уникальных запросов train", queries_train["query_id"].nunique())
print("Пар запрос товар train", len(queries_train))
print("Среднее число оценок на запрос", round(queries_train.groupby("query_id").size().mean(), 1))
display(df_raw[["imt_name", "subj_name", "description"]].sample(5, random_state=42))


---

## Неделя 2 — базовый пайплайн и оценка качества

**Цель:** оперевшись на результаты EDA, создать работающий поиск и получить первые метрики качества.

Прежде чем реализовывать семантический поиск, основанный на эмбеддингах, нужно проверить, как справляется более простая модель, опирающаяся на TF-IDF. Простая модель нужна как baseline, без этой точки отсчёта результат невозможно интерпретировать.


На этой неделе используется только тренировочная разметка. Тестовая выборка оставлена для одного финального прогона в конце проекта.


### 1. Подготовка данных


Реализуйте решения недели 1:

1. Отфильтруйте выбранные категории.
2. Дедуплицируйте строки по `imt_id`. Сформулируйте правило, по которому будете определять, какую строку оставить. К примеру, можете оставить строку с самым информативным описанием. Ответ обоснуйте.
3. Отбросьте карточки, у которых пусты одновременно и `imt_name`, и `description`.
4. Напишите функцию очистки текста.
5. Соберите единый текст карточек в отдельной колонке `product_text`.

In [ ]:
def clean_text(value):
    text = html.unescape(str(value) if pd.notna(value) else "")
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.lower().replace("ё", "е")
    text = re.sub(r"[^a-zа-я0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


category_map = (
    df_raw[["imt_id", "subj_root_name"]]
    .dropna(subset=["subj_root_name"])
    .drop_duplicates("imt_id")
)
relevant_items = queries_train.loc[queries_train["relevance"] >= 2, ["query_id", "item_id"]]
category_scores = (
    relevant_items.merge(category_map, left_on="item_id", right_on="imt_id", how="left")
    .groupby("subj_root_name")["query_id"].nunique()
    .sort_values(ascending=False)
)
selected_categories = category_scores.head(3).index.tolist()
print("Выбранные категории", selected_categories)

filtered = df_raw[df_raw["subj_root_name"].isin(selected_categories)].copy()
filtered["description_length"] = filtered["description"].fillna("").astype(str).str.len()
filtered = filtered.sort_values("description_length", ascending=False).drop_duplicates("imt_id")

for column in ["imt_name", "subj_name", "description"]:
    filtered[column] = filtered[column].map(clean_text)

filtered = filtered[(filtered["imt_name"] != "") | (filtered["description"] != "")].copy()
filtered["product_text"] = (
    filtered["imt_name"] + " . " + filtered["subj_name"] + " . " + filtered["description"]
).str.replace(r"\s+", " ", regex=True).str.strip(" .")

pool_ids = set(assessed_pool["imt_id"])
evaluation_catalog = filtered[filtered["imt_id"].isin(pool_ids)].reset_index(drop=True)
prepared_catalog = filtered.reset_index(drop=True)

print("Полный каталог выбранных категорий", len(prepared_catalog))
print("Каталог оценки", len(evaluation_catalog))
print("Уникальных imt_id", prepared_catalog["imt_id"].nunique())
display(evaluation_catalog[["imt_id", "imt_name", "subj_name", "product_text"]].head())


### Каталог для поиска и оценки

1. Посчитайте метрику при поиске по всему отфильтрованному каталогу. Например, вычислите NDCG@10 лексическим поиском на 100 запросах.
2. Выведите топ-10 для одного запроса и отметьте, какие товары есть в разметке, а каких нет. Выясните, в чём проблема.
3. Ограничьте каталог оценки пулом асессоров: возьмите из `wb_products_dedup.parquet` только список `imt_id` и оставьте в своём подготовленном каталоге пересечение с ним. Тексты и все признаки — ваши, из шага 1.
4. Пересчитайте метрику и сравните результаты.
5. Посмотрите на распределение `relevance` и число оценённых товаров на запрос. Проверьте, у всех ли запросов есть товары с `relevance` ≥ 2.

Для честного сравнения оба метода ранжируют один и тот же каталог оценённых карточек. Товары без оценки для конкретного запроса считаются нерелевантными.


### 2. Модели поиска

Реализуйте два поисковых движка с одинаковым интерфейсом. Это позволит прогнать оба через одну функцию оценки:

In [ ]:
def format_results(scores, top_k):
    order = np.argsort(scores)[::-1][:top_k]
    columns = ["imt_id", "imt_name", "subj_name", "product_text"]
    result = evaluation_catalog.iloc[order][columns].copy()
    result["score"] = np.asarray(scores)[order]
    return result.reset_index(drop=True)



Реализуйте лексический baseline на TF-IDF:

In [ ]:
tfidf = TfidfVectorizer(
    lowercase=False,
    ngram_range=(1, 2),
    min_df=2,
    max_features=60000,
    sublinear_tf=True,
)
tfidf_matrix = tfidf.fit_transform(evaluation_catalog["product_text"])


def search_lexical(query: str, top_k: int = 10) -> pd.DataFrame:
    query_vector = tfidf.transform([clean_text(query)])
    scores = cosine_similarity(query_vector, tfidf_matrix).ravel()
    return format_results(scores, top_k)


print("Размер словаря", len(tfidf.vocabulary_))
display(search_lexical("удобные кроссовки для бега", 5))


Реализуйте семантический поиск:

In [ ]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
semantic_model = SentenceTransformer(MODEL_NAME)

embedding_path = DATA_DIR / "product_embeddings.npy"
if embedding_path.exists():
    product_embeddings = np.load(embedding_path)
else:
    product_embeddings = semantic_model.encode(
        evaluation_catalog["product_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(embedding_path, product_embeddings)

if len(product_embeddings) != len(evaluation_catalog):
    product_embeddings = semantic_model.encode(
        evaluation_catalog["product_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(embedding_path, product_embeddings)


def search_semantic(query: str, top_k: int = 10) -> pd.DataFrame:
    query_embedding = semantic_model.encode(
        [clean_text(query)], normalize_embeddings=True
    )[0]
    scores = product_embeddings @ query_embedding
    return format_results(scores, top_k)


print("Форма матрицы эмбеддингов", product_embeddings.shape)
display(search_semantic("удобные кроссовки для бега", 5))


Прогоните 3–5 запросов вручную и сравните выдачу обоих подходов. Это самый простой способ поймать грубые ошибки до подсчёта метрик.

In [ ]:
example_queries = (
    queries_train.loc[queries_train["item_id"].isin(evaluation_catalog["imt_id"]),
                      ["query_id", "query_text"]]
    .drop_duplicates("query_id")
    .head(5)
)

for query in example_queries["query_text"]:
    print("Запрос", query)
    print("Лексический поиск")
    display(search_lexical(query, 3)[["imt_id", "imt_name", "score"]])
    print("Семантический поиск")
    display(search_semantic(query, 3)[["imt_id", "imt_name", "score"]])


### 3. Оценка качества решений

Реализуйте метрики Precision@K, Recall@K, MRR, NDCG@K, HitRate@K.

Три ключевые особенности, которые надо учесть при расчёте:

1. Порог бинаризации. Precision@K, Recall@K, MRR, HitRate@K бинарны, а шкала `relevance` — от `0` до `3`. С какой оценки товар считается релевантным? Вынесите порог в параметр, а не константу внутри функции.
2. NDCG считайте по полной шкале 0–3 (`gain = 2^rel - 1`), а не по бинаризованной, иначе теряется смысл градуированной разметки.
3. Неразмеченные товары в выдаче трактуйте как нерелевантные (`relevance_map.get(item_id, 0)`).

Выводы запишите в Markdown-ячейке и рабочем файле Markdown.

Порог бинарной релевантности равен 2. NDCG использует полные оценки от 0 до 3. Метрики считаются при K 1, 3, 5 и 10.


In [ ]:
RELEVANCE_THRESHOLD = 2
K_VALUES = [1, 3, 5, 10]

train_filtered = queries_train[
    queries_train["item_id"].isin(evaluation_catalog["imt_id"])
].copy()
query_table = train_filtered[["query_id", "query_text"]].drop_duplicates("query_id")
relevance_by_query = {
    query_id: dict(zip(group["item_id"], group["relevance"]))
    for query_id, group in train_filtered.groupby("query_id")
}


def metrics_at_k(predicted_ids, relevance, k):
    predicted = list(predicted_ids[:k])
    binary = [relevance.get(item_id, 0) >= RELEVANCE_THRESHOLD for item_id in predicted]
    relevant_total = sum(value >= RELEVANCE_THRESHOLD for value in relevance.values())

    precision = sum(binary) / k
    recall = sum(binary) / relevant_total if relevant_total else 0.0
    hit_rate = float(any(binary))
    first_hit = next((rank for rank, hit in enumerate(binary, 1) if hit), None)
    mrr = 1 / first_hit if first_hit else 0.0

    gains = np.array([relevance.get(item_id, 0) for item_id in predicted], dtype=float)
    discounts = np.log2(np.arange(2, k + 2))
    dcg = np.sum((2 ** gains - 1) / discounts)
    ideal = np.sort(np.array(list(relevance.values()), dtype=float))[::-1][:k]
    ideal = np.pad(ideal, (0, max(0, k - len(ideal))))
    idcg = np.sum((2 ** ideal - 1) / discounts)
    ndcg = dcg / idcg if idcg else 0.0
    return precision, recall, mrr, ndcg, hit_rate


def evaluate_search(search_function, model_name):
    rows = []
    for row in query_table.itertuples(index=False):
        predicted = search_function(row.query_text, max(K_VALUES))["imt_id"].tolist()
        relevance = relevance_by_query[row.query_id]
        for k in K_VALUES:
            values = metrics_at_k(predicted, relevance, k)
            rows.append([model_name, row.query_id, k, *values])
    columns = ["model", "query_id", "k", "precision", "recall", "mrr", "ndcg", "hit_rate"]
    return pd.DataFrame(rows, columns=columns)


lexical_metrics = evaluate_search(search_lexical, "TF-IDF")
semantic_metrics = evaluate_search(search_semantic, "MiniLM")
metrics_detailed = pd.concat([lexical_metrics, semantic_metrics], ignore_index=True)
metrics_summary = (
    metrics_detailed.groupby(["model", "k"])[["precision", "recall", "mrr", "ndcg", "hit_rate"]]
    .mean()
    .round(4)
)
display(metrics_summary)

metrics_summary.reset_index().to_csv("week2_metrics.csv", index=False)

comparison = metrics_detailed[metrics_detailed["k"] == 10].pivot(
    index="query_id", columns="model", values="ndcg"
)
comparison["delta_semantic_minus_lexical"] = comparison["MiniLM"] - comparison["TF-IDF"]
best_semantic_ids = comparison.nlargest(3, "delta_semantic_minus_lexical").index
best_lexical_ids = comparison.nsmallest(3, "delta_semantic_minus_lexical").index
display(query_table[query_table["query_id"].isin(best_semantic_ids)].assign(case="semantic better"))
display(query_table[query_table["query_id"].isin(best_lexical_ids)].assign(case="lexical better"))


---

## Неделя 3 — эксперименты, дообучение и MLflow

**Цель:** проверить гипотезы о том, как можно улучшить систему поиска, зафиксировать эксперименты, дать рекомендации бизнесу.

**Главное правило:** «один эксперимент — одна гипотеза — одно изменение». Если поменять сразу модель, шаблон текста и параметры векторизации, прирост будет невозможно объяснить и повторить.

### 1. Эксперименты по улучшению

Проведите минимум три эксперимента. Оформляйте каждый по схеме:

- **Гипотеза** — ваше предположение и его обоснование. Причины «хочу попробовать другую модель» недостаточно, нужны объяснения такого типа: «Модель с большим окном улучшит поиск товаров с длинными описаниями, потому что в EDA я видел, что X% описаний обрезаются по лимиту токенов».
- **Что меняется** — ровно один компонент относительно baseline.
- **Результат** — метрики на тех же данных, с тем же порогом и теми же K.
- **Вывод** — принимаем или отклоняем гипотезу. Что это говорит о данных?

Придумывайте гипотезы, опираясь на ошибки, которые обнаружили на неделе 2. Если выбранная модель путает смежные категории, попробуйте проверить, решит ли проблему добавление данных из `subj_name` в общий текст карточки. Другой пример: если лексический поиск обходит семантический на точных названиях, а семантический выигрывает у лексического, когда дело доходит до неточных формулировок, — это повод попробовать реализовать гибридный поиск.

Направления для разработки гипотез:

- изменить шаблон `product_text`;
- попробовать другую модель эмбеддингов;
- дообучить модели;
- реализовать гибридный поиск с помощью взвешенной суммы скоров или реранкинга семантикой поверх лексического отбора;
- ввести лемматизацию и стоп-слова для TF-IDF;
- изменить параметры векторизации.

In [ ]:
# Эксперимент 1
# Гипотеза:
# Что меняем:


# Ваш код

...

### 2. Финальный тест

Выборка `test` нужна для финальной проверки модели, когда конфигурация окончательно выбрана по результатам валидации. Каждый повторный прогон с подстройкой превращает `test` в ещё одну валидацию и обесценивает финальные показатели.

In [ ]:
RUN_FINAL_TEST_EVAL = False   # Переведите в True, когда конфигурация финальная

# Ваш код


### 3. Трекинг экспериментов в MLflow

Каждая конфигурация — отдельный run с осмысленным именем (`semantic_bge_m3`, а не `run_7`).

Что логировать:
- Все параметры, известные до запуска, — метод, модель, шаблон текста, порог, размер каталога, сид.
- Метрики — все посчитанные значения.
- Артефакты — таблица метрик по запросам, сводная таблица, графики, копия данных, на которых проводилась проверка.

In [ ]:
# Ваш код: подключение к MLflow из .env и логирование прогонов

### 4. Финальные выводы

**4.1. Сводная таблица.** Все подходы на одних данных, с одинаковыми порогом и K. Укажите лучшую конфигурацию и метрики, по которым был сделан вывод.

**4.2. Качественный анализ.** Числа показывают не всю картину. Возьмите запросы, где лучший подход всё ещё ошибается, и опишите характер ошибок.

**4.3. Ограничения оценки.** Перечислите ограничения модели и данных, на которых проводились эксперименты. Например:

- разметка синтетическая и пулированная;
- каталог оценки меньше реального;
- запросы сгенерированы, а не собраны от пользователей.

Этот раздел не ослабляет работу, а усиливает её: специалист, понимающий границы своего решения, вызывает больше доверия.

**4.4. Рекомендация бизнесу.** Скажите заказчику, внедрять ли ему векторный поиск или нет, в каком виде его лучше реализовывать, в каких сценариях он будет помогать, а где может ошибаться.

**4.5. Дальнейшие шаги.** Перечислите возможные действия, которые могли бы улучшить результат:

- использовать реальные логи запросов вместо синтетических;
- перейти на ANN-индекс при росте каталога;
- применить фильтры по атрибутам поверх векторного поиска.

Все возможные шаги обоснуйте.


*Ваши выводы:*
